# Laboratorium 7: Prompting jako eksperyment

W tym laboratorium model pozostaje zamrożony. Zmieniamy prompt i parametry generowania, a następnie mierzymy wpływ tych zmian na jakość i stabilność systemu.

Główne pytanie laboratorium:

**Skąd wiemy, że nowa wersja promptu rzeczywiście poprawia system, a nie tylko wygląda lepiej na kilku przykładach?**


## Po tych zajęciach student potrafi

- zamrozić protokół eksperymentalny dla promptów;
- wersjonować prompty i zapisywać ich skróty SHA-256;
- przeprowadzić kontrolowaną ablację zero-shot;
- porównać zero-shot i few-shot;
- ocenić wpływ kolejności przykładów;
- zmierzyć wpływ temperatury na stabilność;
- porównać odpowiedź w postaci pojedynczej etykiety z odpowiedzią w formacie JSON;
- przygotować mały zestaw trudnych i konfliktowych przypadków;
- analizować błędy promptu;
- oceniać różnice między promptami z przedziałem ufności i potwierdzać wybór na osobnym zbiorze końcowym;
- uzasadnić wybór promptu na podstawie wyników.

In [ ]:
from pathlib import Path
import json
import time
import hashlib
from itertools import permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

RANDOM_STATE = 42
OLLAMA_BASE = 'http://localhost:11434'
ARTIFACTS = Path('../artifacts')
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option('display.max_colwidth', 220)
pd.set_option('display.max_columns', 80)

In [ ]:
def ollama_get(path, timeout=30):
    r = requests.get(f'{OLLAMA_BASE}{path}', timeout=timeout)
    r.raise_for_status()
    return r.json()

def ollama_post(path, payload, timeout=300):
    r = requests.post(
        f'{OLLAMA_BASE}{path}',
        json=payload,
        timeout=timeout,
    )
    r.raise_for_status()
    return r.json()

tags = ollama_get('/api/tags')
models = [x['name'] for x in tags.get('models', [])]

print('Dostępne modele:')
for m in models:
    print('-', m)

MODEL = 'NAZWA_MODELU_Z_LAB_6'   # dokładna nazwa z listy powyżej
assert MODEL in models, f'Model {MODEL} nie jest pobrany w Ollamie.'

# Parametry generowania zamrożone na czas całego eksperymentu.
# num_ctx: zwiększyć, jeśli prompt_eval_count w odpowiedzi zbliża się do tej wartości.
GEN_OPTIONS = {'temperature': 0.0, 'seed': RANDOM_STATE, 'num_ctx': 8192}
print('\nMODEL =', MODEL, '| opcje:', GEN_OPTIONS)

# Zadanie 1: Zamrożenie modelu, zbiorów i protokołu

Proszę wybrać jeden lokalny model z laboratorium 6 i zapisać jego pełną nazwę, kwantyzację oraz parametry generowania (`temperature`, `seed`, `num_ctx`). Model i parametry nie mogą zmieniać się w trakcie eksperymentu (poza zadaniem 6, gdzie zmieniamy temperaturę).

Pracujemy na trzech rozłącznych zbiorach z PolEmo 2.0:

- `dev_set`: 80 przykładów z części `validation`, zbalansowanych według klas; na nim wykonujemy **wszystkie** porównania promptów (zadania 3-8);
- `final_set`: 80 przykładów z części `test`, zbalansowanych według klas; **nie wolno go oglądać ani używać do żadnych decyzji** przed zadaniem 9;
- pula przykładów few-shot -- wyłącznie z części `train`.

Różnice między promptami proszę podawać z 95% przedziałem ufności (bootstrap sparowany po przykładach, jak w laboratorium 2). Przy 80 przykładach błąd standardowy accuracy wynosi ok. 5-6 p.p., więc różnice rzędu kilku punktów mogą być szumem.

Proszę też zmierzyć czas pojedynczego wywołania modelu na przykładowej recenzji. Całe laboratorium wymaga ok. 1000 wywołań, więc proszę oszacować czas jego wykonania na własnym sprzęcie.

In [ ]:
DATA_PATH = ARTIFACTS / 'polemo2_clean.parquet'
assert DATA_PATH.exists(), 'Brak artifacts/polemo2_clean.parquet. Proszę wykonać Lab 1.'

data = pd.read_parquet(DATA_PATH).dropna(subset=['text', 'target']).reset_index(drop=True)
assert set(data['target']) == {'plus', 'minus', 'zero', 'amb'}

train_df = data[data['split'] == 'train'].reset_index(drop=True)
val_df = data[data['split'] == 'validation'].reset_index(drop=True)
test_df = data[data['split'] == 'test'].reset_index(drop=True)

def balanced_sample(df, n, seed):
    per_class = max(1, n // df['target'].nunique())
    parts = [g.sample(min(per_class, len(g)), random_state=seed)
             for _, g in df.groupby('target')]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

dev_set = balanced_sample(val_df, 80, RANDOM_STATE)     # zadania 3-8
final_set = balanced_sample(test_df, 80, RANDOM_STATE)  # tylko zadanie 9

display(dev_set['target'].value_counts())
display(final_set['target'].value_counts())


### Ćwiczenie wprowadzające: który prompt jest lepszy?

Załóżmy, że porównujemy dwa prompty klasyfikujące opinie z PolEmo 2.0.

Oba prompty przetestowano na **tych samych 80 recenzjach**: po 20 przykładów z każdej klasy (`plus`, `minus`, `zero`, `amb`).

Uzyskano następujące wyniki:

| Metryka | Prompt A | Prompt B |
|---|---:|---:|
| Accuracy | **75%** | 70% |
| Macro-F1 | 0.667 | **0.700** |
| F1 `plus` | 1.00 | 0.70 |
| F1 `minus` | 1.00 | 0.70 |
| F1 `zero` | 0.67 | 0.70 |
| F1 `amb` | 0.00 | 0.70 |

**Co się wydarzyło?**

- **Prompt A** poprawnie sklasyfikował wszystkie przykłady `plus`, `minus` i `zero`, ale wszystkie 20 przykładów `amb` błędnie uznał za `zero`.
- **Prompt B** poprawnie sklasyfikował po 14 przykładów każdej klasy. Pozostałe 6 przykładów z każdej klasy przypisał błędnie do innej klasy (cyklicznie: `plus → minus → zero → amb → plus`).

#### Pytania do dyskusji

1. Który prompt ma wyższą ogólną skuteczność?
2. Dlaczego macro-F1 wskazuje na inny prompt niż accuracy?
3. Który prompt wybraliby Państwo do systemu analizującego opinie klientów? Proszę uzasadnić wybór.
4. Czy te wyniki wystarczają, aby stwierdzić, że jeden prompt jest rzeczywiście lepszy? Jakich informacji jeszcze potrzebujemy?

**Wniosek do zapamiętania:** wybór promptu zależy nie tylko od wartości metryki, ale również od rozkładu błędów, celu systemu i niepewności pomiaru.

# Zadanie 2: Rejestr promptów

Proszę zaimplementować rejestr promptów zawierający dla każdej wersji:

- identyfikator;
- numer wersji;
- treść instrukcji;
- datę;
- krótki opis zmiany;
- SHA-256 pełnej treści promptu (łącznie z ewentualnymi przykładami few-shot).

Każdy wynik benchmarku musi być powiązany z konkretnym skrótem SHA-256.

SHA-256 identyfikuje kanoniczny szablon promptu wraz z przykładami few-shot, a nie każdą wiadomość zawierającą aktualnie klasyfikowaną recenzję.

In [ ]:
# Proszę zaimplementować rejestr promptów i funkcję wyliczającą SHA-256.

# Zadanie 3: Ablacja zero-shot

Proszę przygotować trzy wersje promptu:

1. minimalną instrukcję;
2. instrukcję z listą klas;
3. instrukcję z krótkim opisem znaczenia klas.

Dla każdego wariantu proszę policzyć accuracy, macro-F1, F1 dla każdej klasy, odsetek niepoprawnych odpowiedzi i średni czas.

W prompcie klasy warto nazwać po polsku (np. pozytywny, negatywny, neutralny, niejednoznaczny), a odpowiedź modelu odwzorować z powrotem na `plus`, `minus`, `zero`, `amb`. Odpowiedź spoza tego zbioru klas (po normalizacji wielkości liter i białych znaków) uznajemy za niepoprawną: liczymy ją jako błędną predykcję (w sklearn: `f1_score(..., labels=CLASSES)`), a jej odsetek raportujemy osobno.

Wariant najlepszy wybieramy na `dev_set`. Różnice między wariantami proszę podać z 95% przedziałem ufności.

Zmieniać wolno wyłącznie treść promptu.

HINT: Warianty powinny być budowane przyrostowo: instrukcja podstawowa, ta sama instrukcja z listą klas, a następnie z definicjami klas.

In [ ]:
# Proszę zaimplementować wspólną funkcję uruchamiającą model i benchmark trzech wersji zero-shot.

# Zadanie 4: Few-shot

Dla najlepszego promptu zero-shot proszę porównać:

- zero-shot;
- 1 przykład na klasę;
- 2 przykłady na klasę.

Przykłady few-shot muszą pochodzić z części `train` (nie z `dev_set` ani `final_set`). Proszę wybrać je deterministycznie (np. z `random_state`), zapisać ich identyfikatory i użyć tego samego zestawu dla całego testu danego wariantu. Recenzje w PolEmo są długie, więc warto wybierać przykłady o ograniczonej długości (np. do ok. 600 znaków).

Proszę sprawdzić pole `prompt_eval_count` w odpowiedzi Ollamy i upewnić się, że prompt (zwłaszcza 2 przykłady na klasę) nie został obcięty przez okno kontekstu (`num_ctx`).

Proszę zmierzyć zmianę macro-F1, czasu oraz długości promptu.

In [ ]:
# Proszę zaimplementować benchmark zero-shot vs 1-shot vs 2-shot na klasę.

# Zadanie 5: Wrażliwość na kolejność demonstracji

Proszę użyć zestawu 2 przykładów na klasę z zadania 4 i ułożyć go w trzech kolejnościach (np. pogrupowane według klas, kolejność odwrócona, kolejność losowa z ustalonym `seed`).

Dla każdej kolejności proszę uruchomić ten sam `dev_set` i porównać macro-F1 oraz odsetek rekordów, dla których zmieniła się predykcja. Odsetek zmian liczymy dla każdej pary kolejności (3 pary) i podajemy średnią oraz maksimum.

Celem jest sprawdzenie, czy semantycznie równoważny prompt daje stabilne zachowanie.

In [ ]:
# Proszę przeprowadzić eksperyment z trzema kolejnościami przykładów.

# Zadanie 6: Temperatura i stabilność

Proszę porównać temperaturę `0.0` i `0.7`.

Każdą konfigurację należy uruchomić trzy razy na tej samej mniejszej próbce: 40 przykładów z `dev_set` (po 10 na klasę). Przy `temperature = 0.7` każde powtórzenie powinno mieć inny `seed` (np. 1, 2, 3) -- przy stałym `seed` Ollama zwraca powtarzalne wyniki i nie zmierzymy niestabilności. Przy `temperature = 0.0` proszę użyć stałego `seed`.

Proszę policzyć:

- średnie macro-F1;
- odchylenie standardowe z trzech powtórzeń (z próby, ddof=1);
- procent przykładów, których predykcja zmienia się między powtórzeniami;
- odsetek błędów formatu.

Przy temperaturze 0.0 oczekujemy odchylenia bliskiego zeru; brak zmienności również jest wynikiem.

In [ ]:
# Proszę przeprowadzić kontrolowany test temperatury i stabilności.

# Zadanie 7: Etykieta kontra JSON

Dla najlepszego dotychczasowego promptu proszę porównać dwa formaty odpowiedzi:

1. pojedyncza etykieta;
2. JSON zgodny z jawnie określonym schematem (np. `{"label": "plus"}`), wymagany samym poleceniem w prompcie, **bez** parametru `format` Ollamy.

Gdybyśmy użyli parametru `format` ze schematem JSON, serwer wymuszałby składnię, a JSON validity byłaby trywialna; dlatego tego parametru tu nie stosujemy.

Proszę mierzyć poprawność klasy, JSON validity, schema validity, długość odpowiedzi i czas.

Proszę sprawdzić, czy wymuszenie struktury zmienia jakość samej klasyfikacji (różnica macro-F1 z przedziałem ufności).

*(opcjonalne)* Proszę powtórzyć wariant 2 z parametrem `format` ze schematem JSON i porównać poprawność formatu, jakość i czas.

In [ ]:
# Proszę przygotować benchmark plain label vs JSON.

# Zadanie 8: Analiza 12 przypadków

Proszę przygotować dokładnie 12 przypadków do analizy:

- 6 rzeczywistych błędów wybranego promptu na `dev_set` (różnych rodzajów);
- 6 przypadków skonstruowanych samodzielnie, obejmujących m.in.: negację, klasę `amb`, bardzo krótki tekst, nietypową interpunkcję oraz co najmniej dwa teksty zawierające instrukcję sprzeczną z promptem, np. „zignoruj wcześniejsze instrukcje” (tzw. wstrzyknięcie instrukcji, ang. *prompt injection*).

Dla wybranych przypadków proszę dodatkowo sprawdzić, czy wynik zmienia się po zmianie kolejności przykładów lub formatu odpowiedzi (etykieta / JSON).

Tabela:

| nr | przypadek | prompt / wariant | oczekiwane | odpowiedź modelu | typ problemu | komentarz |
|---:|---|---|---|---|---|---|

Proszę wskazać, które błędy są stabilne między wersjami promptu, a które pojawiają się dopiero po zmianie instrukcji.

In [ ]:
# Proszę zbudować i przeanalizować zestaw 12 trudnych przypadków.

# Zadanie 9: Ocena końcowa i interpretacja wyników

**Ocena końcowa.** Zbiór `final_set` dotąd nie był używany. Proszę jednokrotnie ocenić na nim dwa prompty: bazowy (minimalna instrukcja zero-shot) i wybrany jako najlepszy na `dev_set`. Po obejrzeniu wyników nie wolno zmieniać promptów. Proszę podać macro-F1 obu promptów oraz 95% przedział ufności ich różnicy (bootstrap sparowany po przykładach) i porównać go z różnicą na `dev_set`.

Następnie proszę odpowiedzieć:

1. Która zmiana promptu dała największą poprawę i czy była ona stabilna?
2. Czy few-shot pomógł wszystkim klasom jednakowo?
3. Jak duża była wrażliwość na kolejność przykładów?
4. Czy temperatura 0.7 przyniosła korzyść w zadaniu klasyfikacyjnym?
5. Które błędy promptu wyglądają na problem modelu, a które na problem instrukcji?
6. Czy poprawa zmierzona na `dev_set` utrzymała się na `final_set`? Co to mówi o wyborze promptu na podstawie jednego zbioru?

HINT: Losujemy wspólne indeksy obserwacji dla porównywanych promptów. Przy 80 przykładach na zbiór wyniki mogą być bardzo niepewne, zwłaszcza macro-F1 poszczególnych klas. Zbalansowany `final_set` jest dobry do porównań diagnostycznych, ale nie odzwierciedla automatycznie jakości przy naturalnym rozkładzie klas.

In [ ]:
# Proszę przeprowadzić jednokrotną ocenę końcową na final_set.

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wskazać prompt, który zostałby wdrożony, i uzasadnić wybór na podstawie jakości, stabilności, kosztu, analizy 12 przypadków oraz oceny końcowej na final_set.

Proszę także wskazać jeden wariant odrzucony i wyjaśnić, dlaczego wyższy wynik w pojedynczej metryce nie wystarczył do jego wyboru.

# Karta promptu

Proszę utworzyć `artifacts/prompt_card.md` zawierający: model i parametry, wersję i hash wybranego promptu, zbiór ewaluacyjny, format odpowiedzi, wyniki zero-shot/few-shot, stabilność, błędy formatu, wynik oceny końcowej na final_set, obserwacje z 12 przypadków, przeznaczenie, ograniczenia i uzasadnienie wyboru.

# Wnioski

Proszę zapisać od pięciu do ośmiu krótkich wniosków wynikających z eksperymentu. Powinny dotyczyć konstrukcji promptu, few-shot, kolejności, temperatury, struktury odpowiedzi i stabilności.


# Warunki zaliczenia laboratorium

- [ ] jeden zamrożony model;
- [ ] stałe zbiory `dev_set` i `final_set` (po 80 przykładów) oraz pula few-shot z `train`;
- [ ] rejestr promptów i SHA-256;
- [ ] trzy warianty zero-shot;
- [ ] przedziały ufności dla różnic między promptami;
- [ ] zero-shot vs 1-shot vs 2-shot na klasę;
- [ ] trzy kolejności przykładów;
- [ ] temperatura 0.0 vs 0.7 z powtórzeniami;
- [ ] etykieta vs JSON;
- [ ] tabela 12 przypadków;
- [ ] jednokrotna ocena końcowa na `final_set` i odpowiedzi do zadań 9 i 10;
- [ ] `artifacts/prompt_card.md`.